# 1. Load Data from all the stock csv files in directory

In [5]:
import datetime
import glob
import numpy as np
import pandas as pd

# Get CSV files list from a folder
csv_files = glob.glob("./TSLA-BID_ASK-*.csv")

# Read each CSV file into DataFrame
# This creates a list of dataframes
df_list = (pd.read_csv(file) for file in csv_files)

# Concatenate all DataFrames
df   = pd.concat(df_list, ignore_index=True)

## 1.1 Drop duplicates & check for missing values

In [14]:
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("Before: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

Before: # of duplicates 0  out of  1908010  or  0.0 %


In [11]:
# Drop duplicate entries
df.drop_duplicates(subset=['date'], keep='first', inplace=True)

In [13]:
# verify there are no duplicate values
df["date"].is_unique
dups = len(df['date'])-len(df['date'].drop_duplicates())
print("After: # of duplicates", dups, ' out of ', len(df) , ' or ', round(dups/len(df),5), '%')

After: # of duplicates 0  out of  1908010  or  0.0


(None, '%')

## 1.2 Print out all dates with confirming # of quotes (23400) and non-conforming number of quotes

In [15]:
df['date2_str']= df['date'][::].str.slice(stop=10)
pd_group_cnt = df.groupby(['date2_str'])['date2_str'].count().to_frame()
print("Confirming / correct number of quotes: 23,400")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']==23400] )
pd.set_option('display.max_rows', None)
print("Dates Missing quotes")
print(pd_group_cnt.loc[pd_group_cnt['date2_str']!=23400] )
pd.set_option('display.max_rows', 10)
df = df.drop('date2_str', axis=1)

Confirming / correct number of quotes: 23,400
            date2_str
date2_str            
2022-06-23      23400
2022-06-24      23400
2022-06-27      23400
2022-06-28      23400
2022-06-29      23400
...               ...
2022-10-11      23400
2022-10-12      23400
2022-10-13      23400
2022-10-14      23400
2022-10-17      23400

[81 rows x 1 columns]
Dates Missing quotes
            date2_str
date2_str            
2022-06-22      12610


In [16]:
# Only want to track average to 3 decimal places.  Otherwise, end up with a lot of digits
df['average'] = df['average'].round(decimals = 3)

# 2.0 Describe data

In [17]:
df = df.sort_index(ascending=True)
#df = df.tail(10000)
#df = df.tail(30000)
df.describe()

,open,high,low,close,volume,average,barCount
count,1.908010e+06,1.908010e+06,1.908010e+06,1.908010e+06,1908010.0,1908010.0,1908010.0
mean,2.686557e+02,2.687600e+02,2.686341e+02,2.687381e+02,-1.0,-1.0,-1.0
std,2.991326e+01,2.991467e+01,2.991367e+01,2.991502e+01,0.0,0.0,0.0
min,2.041700e+02,2.042100e+02,2.041500e+02,2.041900e+02,-1.0,-1.0,-1.0
25%,2.401394e+02,2.402400e+02,2.401167e+02,2.402200e+02,-1.0,-1.0,-1.0
50%,2.755700e+02,2.756600e+02,2.755500e+02,2.756400e+02,-1.0,-1.0,-1.0
75%,2.950333e+02,2.951500e+02,2.950200e+02,2.951290e+02,-1.0,-1.0,-1.0
max,3.145250e+02,3.147100e+02,3.144167e+02,3.146263e+02,-1.0,-1.0,-1.0


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1908010 entries, 0 to 1925999
Data columns (total 9 columns):
 #   Column     Dtype  
---  ------     -----  
 0   date       object 
 1   open       float64
 2   high       float64
 3   low        float64
 4   close      float64
 5   volume     float64
 6   average    float64
 7   barCount   int64  
 8   date2_str  object 
dtypes: float64(6), int64(1), object(2)
memory usage: 145.6+ MB


# 3.0 Add computed columns

In [20]:
df.set_index('date')
df = df.sort_index()

In [21]:
# lambda functions

#return 1st value in series
def firstValue(rows):
    return rows.iloc[0]

#return last value in series
def lastValue(rows):
    return rows.iloc[-1]

#return arrow indicator for boxed in values;
#   -1 below lower bound
#    0 inside the box
#   +1 above the max value
def arrow(new_amt, old_amt, box):
    if old_amt == np.nan:
        return np.nan
    if new_amt == np.nan:
        return np.nan
    if (new_amt - old_amt) <= (box * -1):
        return '-1'
    if (new_amt - old_amt) >= box:
        return '1'
    else:
        return '0'

#  lambda function to adds up the last 5 values, excluding the very last value
def sum_last_5(rows):
    #print ("[" , rows[-6:-1], rows[-6:-1].sum(), "]")
    return rows[-6:-1].sum()

#  lambda function returns lowest of the last 5 values, excluding the very last value
def min_last_5(rows):
    return rows.iloc[-6:-1].min()

#  lambda function returns higest of  the last 5 values, excluding the very last value
def max_last_5(rows):
    return rows.iloc[-6:-1].max()

# Store/Save 1 second windows for h1, h2, h3, h4, h5

In [22]:
%%time
df['h1s_high_max'] = df['high'].rolling(window=2).agg( {'maxLast': firstValue})
df['h1s_low_min'] = df['low'].rolling(window=2).agg( {'minLast': firstValue})
df['h1s_barCount_sum'] = df['barCount'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_volume_sum'] = df['volume'].rolling(window=2).agg( {'sumLast': firstValue})
df['h1s_average_avg'] = df['average'].rolling(window=2).agg( {'sumLast': firstValue})

In [23]:
%%time
df['h2s_high_max'] = df['high'].rolling(window=3).agg( {'maxLast': firstValue})
df['h2s_low_min'] = df['low'].rolling(window=3).agg( {'minLast': firstValue})
df['h2s_barCount_sum'] = df['barCount'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_volume_sum'] = df['volume'].rolling(window=3).agg( {'sumLast': firstValue})
df['h2s_average_avg'] = df['average'].rolling(window=3).agg( {'sumLast': firstValue})


In [24]:
%%time
df['h3s_high_max'] = df['high'].rolling(window=4).agg( {'maxLast': firstValue})
df['h3s_low_min'] = df['low'].rolling(window=4).agg( {'minLast': firstValue})
df['h3s_barCount_sum'] = df['barCount'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_volume_sum'] = df['volume'].rolling(window=4).agg( {'sumLast': firstValue})
df['h3s_average_avg'] = df['average'].rolling(window=4).agg( {'sumLast': firstValue})


In [25]:
%%time
df['h4s_high_max'] = df['high'].rolling(window=5).agg( {'maxLast': firstValue})
df['h4s_low_min'] = df['low'].rolling(window=5).agg( {'minLast': firstValue})
df['h4s_barCount_sum'] = df['barCount'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_volume_sum'] = df['volume'].rolling(window=5).agg( {'sumLast': firstValue})
df['h4s_average_avg'] = df['average'].rolling(window=5).agg( {'sumLast': firstValue})

In [26]:
df = df.drop('date2_str', axis=1)

## Compute 5 second window summary

In [ ]:
%%time
df['h5s_high_max'] = df['high'].rolling(window=6).agg( {'maxLast5': max_last_5})
df['h5s_low_min'] = df['low'].rolling(window=6).agg( {'minLast5': min_last_5})
df['h5s_barCount_sum'] = df['barCount'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['h5s_volume_sum'] = df['volume'].rolling(window=6).agg( {'sumLast5': sum_last_5})
df['_h5s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=6).agg({'SumLast5': sum_last_5})

df['h5s_average_avg'] = df['_h5s_weighted_vol_avg_sum'] / df['h5s_volume_sum']
df['h5s_average_avg'] = df['h5s_average_avg'].round(decimals = 3)
df.drop(columns=['_h5s_weighted_vol_avg_sum'])
#

## Compute 10 second window summary

In [ ]:
%%time
df['h10s_high_max'] = df['high'].rolling(window=11).agg( {'maxLast5': max_last_5})
df['h10s_low_min'] = df['low'].rolling(window=11).agg( {'minLast5': min_last_5})
df['h10s_barCount_sum'] = df['barCount'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['h10s_volume_sum'] = df['volume'].rolling(window=11).agg( {'sumLast5': sum_last_5})
df['_h10s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=11).agg({'SumLast5': sum_last_5})

df['h10s_average_avg'] = df['_h10s_weighted_vol_avg_sum'] / df['h10s_volume_sum']
df['h10s_average_avg'] = df['h10s_average_avg'].round(decimals = 3)
df.drop(columns=['_h10s_weighted_vol_avg_sum'])

## Compute 15 second window summary

In [ ]:
%%time
df['h15s_high_max'] = df['high'].rolling(window=16).agg( {'maxLast5': max_last_5})
df['h15s_low_min'] = df['low'].rolling(window=16).agg( {'minLast5': min_last_5})
df['h15s_barCount_sum'] = df['barCount'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['h15s_volume_sum'] = df['volume'].rolling(window=16).agg( {'sumLast5': sum_last_5})
df['_h15s_weighted_vol_avg_sum'] = df['_weighted_vol_avg'].rolling(window=16).agg({'SumLast5': sum_last_5})

df['h15s_average_avg'] = df['_h15s_weighted_vol_avg_sum'] / df['h15s_volume_sum']
df['h15s_average_avg'] = df['h15s_average_avg'].round(decimals = 3)
df.drop(columns=['_h15s_weighted_vol_avg_sum'])

In [27]:
%%time
df = df.drop(columns=['_weighted_vol_avg'])

KeyError: "['_weighted_vol_avg'] not found in axis"

In [28]:
df.set_index('date')
df = df.sort_index(ascending=False)
df.reset_index()
df.head(100)

,date,open,high,low,close,volume,average,barCount,h1s_high_max,h1s_low_min,...,h3s_high_max,h3s_low_min,h3s_barCount_sum,h3s_volume_sum,h3s_average_avg,h4s_high_max,h4s_low_min,h4s_barCount_sum,h4s_volume_sum,h4s_average_avg
1925999,2022-08-02 15:29:50,303.1933,303.2933,303.1900,303.2900,-1.0,-1.0,-1,303.2933,303.1900,...,303.2933,303.2067,-1.0,-1.0,-1.0,303.2933,303.2433,-1.0,-1.0,-1.0
1925998,2022-08-02 15:29:49,303.1903,303.2933,303.1900,303.2933,-1.0,-1.0,-1,303.2933,303.2067,...,303.2933,303.2433,-1.0,-1.0,-1.0,303.2933,303.1367,-1.0,-1.0,-1.0
1925997,2022-08-02 15:29:48,303.2330,303.2933,303.2067,303.2900,-1.0,-1.0,-1,303.2933,303.2067,...,303.2933,303.1367,-1.0,-1.0,-1.0,303.2500,303.1367,-1.0,-1.0,-1.0
1925996,2022-08-02 15:29:47,303.2300,303.2933,303.2067,303.2933,-1.0,-1.0,-1,303.2933,303.2433,...,303.2500,303.1367,-1.0,-1.0,-1.0,303.2500,303.1367,-1.0,-1.0,-1.0
1925995,2022-08-02 15:29:46,303.2433,303.2933,303.2433,303.2933,-1.0,-1.0,-1,303.2933,303.1367,...,303.2500,303.1367,-1.0,-1.0,-1.0,303.2500,303.0767,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1925904,2022-08-02 15:28:15,302.8063,302.9067,302.7733,302.9000,-1.0,-1.0,-1,302.9100,302.7667,...,302.9900,302.7700,-1.0,-1.0,-1.0,302.9933,302.8867,-1.0,-1.0,-1.0
1925903,2022-08-02 15:28:14,302.7733,302.9100,302.7667,302.8897,-1.0,-1.0,-1,302.9167,302.8333,...,302.9933,302.8867,-1.0,-1.0,-1.0,303.0000,302.8867,-1.0,-1.0,-1.0
1925902,2022-08-02 15:28:13,302.8333,302.9167,302.8333,302.9100,-1.0,-1.0,-1,302.9900,302.7700,...,303.0000,302.8867,-1.0,-1.0,-1.0,303.0000,302.8867,-1.0,-1.0,-1.0
1925901,2022-08-02 15:28:12,302.8323,302.9900,302.7700,302.8937,-1.0,-1.0,-1,302.9933,302.8867,...,303.0000,302.8867,-1.0,-1.0,-1.0,303.0000,302.9133,-1.0,-1.0,-1.0


## Compute Future 5 second window summary

In [29]:
%%time
df['f5s_average'] = df['average'].rolling(window=6).agg( {'firstValue': firstValue})
df.head(100)

,date,open,high,low,close,volume,average,barCount,h1s_high_max,h1s_low_min,...,h3s_low_min,h3s_barCount_sum,h3s_volume_sum,h3s_average_avg,h4s_high_max,h4s_low_min,h4s_barCount_sum,h4s_volume_sum,h4s_average_avg,f5s_average
1925999,2022-08-02 15:29:50,303.1933,303.2933,303.1900,303.2900,-1.0,-1.0,-1,303.2933,303.1900,...,303.2067,-1.0,-1.0,-1.0,303.2933,303.2433,-1.0,-1.0,-1.0,NaN
1925998,2022-08-02 15:29:49,303.1903,303.2933,303.1900,303.2933,-1.0,-1.0,-1,303.2933,303.2067,...,303.2433,-1.0,-1.0,-1.0,303.2933,303.1367,-1.0,-1.0,-1.0,NaN
1925997,2022-08-02 15:29:48,303.2330,303.2933,303.2067,303.2900,-1.0,-1.0,-1,303.2933,303.2067,...,303.1367,-1.0,-1.0,-1.0,303.2500,303.1367,-1.0,-1.0,-1.0,NaN
1925996,2022-08-02 15:29:47,303.2300,303.2933,303.2067,303.2933,-1.0,-1.0,-1,303.2933,303.2433,...,303.1367,-1.0,-1.0,-1.0,303.2500,303.1367,-1.0,-1.0,-1.0,NaN
1925995,2022-08-02 15:29:46,303.2433,303.2933,303.2433,303.2933,-1.0,-1.0,-1,303.2933,303.1367,...,303.1367,-1.0,-1.0,-1.0,303.2500,303.0767,-1.0,-1.0,-1.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1925904,2022-08-02 15:28:15,302.8063,302.9067,302.7733,302.9000,-1.0,-1.0,-1,302.9100,302.7667,...,302.7700,-1.0,-1.0,-1.0,302.9933,302.8867,-1.0,-1.0,-1.0,-1.0
1925903,2022-08-02 15:28:14,302.7733,302.9100,302.7667,302.8897,-1.0,-1.0,-1,302.9167,302.8333,...,302.8867,-1.0,-1.0,-1.0,303.0000,302.8867,-1.0,-1.0,-1.0,-1.0
1925902,2022-08-02 15:28:13,302.8333,302.9167,302.8333,302.9100,-1.0,-1.0,-1,302.9900,302.7700,...,302.8867,-1.0,-1.0,-1.0,303.0000,302.8867,-1.0,-1.0,-1.0,-1.0
1925901,2022-08-02 15:28:12,302.8323,302.9900,302.7700,302.8937,-1.0,-1.0,-1,302.9933,302.8867,...,302.8867,-1.0,-1.0,-1.0,303.0000,302.9133,-1.0,-1.0,-1.0,-1.0


In [30]:
df = df.sort_index(ascending=True)
df.reset_index()
df.head(100)

,date,open,high,low,close,volume,average,barCount,h1s_high_max,h1s_low_min,...,h3s_low_min,h3s_barCount_sum,h3s_volume_sum,h3s_average_avg,h4s_high_max,h4s_low_min,h4s_barCount_sum,h4s_volume_sum,h4s_average_avg,f5s_average
0,2022-08-25 12:29:54,293.780,293.81,293.78,293.810,-1.0,-1.0,-1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
1,2022-08-25 12:29:55,293.790,293.89,293.76,293.889,-1.0,-1.0,-1,293.81,293.78,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
2,2022-08-25 12:29:56,293.791,293.98,293.79,293.960,-1.0,-1.0,-1,293.89,293.76,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0
3,2022-08-25 12:29:57,293.839,293.96,293.79,293.960,-1.0,-1.0,-1,293.98,293.79,...,293.78,-1.0,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,-1.0
4,2022-08-25 12:29:58,293.919,293.96,293.84,293.960,-1.0,-1.0,-1,293.96,293.79,...,293.76,-1.0,-1.0,-1.0,293.81,293.78,-1.0,-1.0,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2022-08-25 12:31:29,293.490,293.55,293.49,293.550,-1.0,-1.0,-1,293.55,293.49,...,293.58,-1.0,-1.0,-1.0,293.71,293.57,-1.0,-1.0,-1.0,-1.0
96,2022-08-25 12:31:30,293.469,293.57,293.42,293.569,-1.0,-1.0,-1,293.55,293.49,...,293.50,-1.0,-1.0,-1.0,293.71,293.58,-1.0,-1.0,-1.0,-1.0
97,2022-08-25 12:31:31,293.509,293.65,293.47,293.609,-1.0,-1.0,-1,293.57,293.42,...,293.49,-1.0,-1.0,-1.0,293.65,293.50,-1.0,-1.0,-1.0,-1.0
98,2022-08-25 12:31:32,293.559,293.61,293.51,293.610,-1.0,-1.0,-1,293.65,293.47,...,293.49,-1.0,-1.0,-1.0,293.55,293.49,-1.0,-1.0,-1.0,-1.0


In [31]:
%%time
pd.set_option('display.max_rows', 100)

df['f5s_10c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.10), axis=1)
df['f5s_15c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.15), axis=1)
df['f5s_20c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.20), axis=1)
df['f5s_25c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.25), axis=1)
df['f5s_30c_arrow'] = df.apply(lambda x: arrow(x['f5s_average'], x['average'], 0.30), axis=1)


CPU times: user 1min 7s, sys: 2.95 s, total: 1min 10s
Wall time: 1min 10s


In [34]:
df.head(100)

CPU times: user 70 µs, sys: 0 ns, total: 70 µs
Wall time: 71.8 µs


,date,open,high,low,close,volume,average,barCount,h1s_high_max,h1s_low_min,...,h4s_low_min,h4s_barCount_sum,h4s_volume_sum,h4s_average_avg,f5s_average,f5s_10c_arrow,f5s_15c_arrow,f5s_20c_arrow,f5s_25c_arrow,f5s_30c_arrow
0,2022-08-25 12:29:54,293.780,293.81,293.78,293.810,-1.0,-1.0,-1,NaN,NaN,...,NaN,NaN,NaN,NaN,-1.0,0,0,0,0,0
1,2022-08-25 12:29:55,293.790,293.89,293.76,293.889,-1.0,-1.0,-1,293.81,293.78,...,NaN,NaN,NaN,NaN,-1.0,0,0,0,0,0
2,2022-08-25 12:29:56,293.791,293.98,293.79,293.960,-1.0,-1.0,-1,293.89,293.76,...,NaN,NaN,NaN,NaN,-1.0,0,0,0,0,0
3,2022-08-25 12:29:57,293.839,293.96,293.79,293.960,-1.0,-1.0,-1,293.98,293.79,...,NaN,NaN,NaN,NaN,-1.0,0,0,0,0,0
4,2022-08-25 12:29:58,293.919,293.96,293.84,293.960,-1.0,-1.0,-1,293.96,293.79,...,293.78,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0
5,2022-08-25 12:29:59,293.840,293.96,293.84,293.960,-1.0,-1.0,-1,293.96,293.84,...,293.76,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0
6,2022-08-25 12:30:00,293.850,293.96,293.84,293.950,-1.0,-1.0,-1,293.96,293.84,...,293.79,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0
7,2022-08-25 12:30:01,293.850,293.95,293.85,293.950,-1.0,-1.0,-1,293.96,293.84,...,293.79,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0
8,2022-08-25 12:30:02,293.860,293.95,293.85,293.950,-1.0,-1.0,-1,293.95,293.85,...,293.84,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0
9,2022-08-25 12:30:03,293.791,293.95,293.79,293.910,-1.0,-1.0,-1,293.95,293.85,...,293.84,-1.0,-1.0,-1.0,-1.0,0,0,0,0,0


In [33]:
df.to_csv("./TSLA-BID_ASK.csv", index=False)